In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB

# 1. Đọc dữ liệu đồng bộ với phần của Hồng Anh
try:
    df = pd.read_csv('D00 - wine.data.csv')
    X = df.drop('Class', axis=1)
    y = df['Class']
except FileNotFoundError:
    from sklearn.datasets import load_wine
    wine = load_wine()
    X, y = wine.data, wine.target

# 2. Xây dựng Pipeline (StandardScaler + GaussianNB) tránh rò rỉ dữ liệu
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('gnb', GaussianNB())
])

# 3. Tối ưu tham số với GridSearchCV và 10-fold StratifiedKFold
param_grid = {'gnb__var_smoothing': np.logspace(0, -9, num=100)}
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='f1_weighted',
    return_train_score=True,
    n_jobs=-1
)

grid_search.fit(X, y)

print("--- KẾT QUẢ TỐI ƯU THAM SỐ (GRIDSEARCHCV) ---")
print("Tham số var_smoothing tốt nhất:", grid_search.best_params_['gnb__var_smoothing'])
print(f"F1-score weighted cao nhất (CV): {grid_search.best_score_:.4f}")

# 4. Đánh giá 10-fold cross-validation với mô hình tốt nhất
best_gnb_model = grid_search.best_estimator_
f1_scores_10fold = cross_val_score(best_gnb_model, X, y, cv=cv_strategy, scoring='f1_weighted')

print("\n--- CHI TIẾT 10-FOLD CROSS-VALIDATION ---")
for i, score in enumerate(f1_scores_10fold, 1):
    print(f"Fold {i:2d}: F1-score = {score:.4f}")

print(f"\nF1-score Trung bình: {f1_scores_10fold.mean():.4f}")
print(f"Độ lệch chuẩn (Std): {f1_scores_10fold.std():.4f}")